# FAIRe Metadata Filler
**UK Centre for Ecology & Hydrology (UKCEH)**

Fills the [FAIRe eDNA metadata template](https://fair-edna.github.io/download.html) from your project report.

1. *Runtime → Run all* (or Ctrl+F9) and upload your **report** when asked (`.doc`, `.docx`, `.pdf` or `.txt`).
2. The official **FAIRe FULL template** is used automatically (tick the box in Step 3 to upload your own).
3. In Step 5 most values are **accepted automatically**; only the ones that need a look are listed, each with the sentence it came from. Correct anything in Step 6.
   Optional: upload a saved **profile** in Step 3 to re-apply corrections from earlier reports, and save a new one in Step 6.
4. Download the filled workbook from the last cell.

🔒 By default the report is read only inside this Colab session. Claude is used only if you switch it on in Step 4 for a non-confidential report.

---

In [ ]:
#@title ⚙️  Step 1 — Install dependencies
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pypdf', 'anthropic'])
print('✅  Dependencies ready.')

In [ ]:
#@title 📥  Step 2 — Download the latest tool code from GitHub
import urllib.request, importlib.util
REPO = 'https://github.com/UKCEH-MolecularEcology/fair-edna-filler/raw/main'
urllib.request.urlretrieve(f'{REPO}/server.py', 'faire_server.py')    # always fetch the latest version
spec = importlib.util.spec_from_file_location('faire', 'faire_server.py')
faire = importlib.util.module_from_spec(spec); spec.loader.exec_module(faire)
print(f'✅  Tool code downloaded (FAIRe template v{faire.TEMPLATE_VERSION}).')

In [ ]:
#@title 📂  Step 3 — Your report and the FAIRe template
#@markdown The official FAIRe FULL template is downloaded automatically.
#@markdown Tick this box only if you want to upload your own template instead (e.g. made with FAIRe-ator):
use_my_own_template = False #@param {type:"boolean"}
#@markdown Tick to upload a profile of corrections saved from earlier reports (a `.json` file):
use_a_profile = False #@param {type:"boolean"}
from google.colab import files as colab_files
import pathlib, urllib.request

print('Upload your REPORT (.doc / .docx / .pdf / .txt):')
up = colab_files.upload()
report_path = pathlib.Path(list(up.keys())[0])
print(f'  ✅  Report: {report_path.name}')

if use_my_own_template:
    print('\nUpload your FAIRe TEMPLATE (.xlsx):')
    up = colab_files.upload()
    template_path = pathlib.Path(list(up.keys())[0])
else:
    template_path = pathlib.Path(faire.TEMPLATE_FILE)
    urllib.request.urlretrieve(f'{REPO}/template/{faire.TEMPLATE_FILE}', template_path)
print(f'  ✅  Template: {template_path.name}' + ('' if use_my_own_template else ' (official FAIRe FULL template)'))
profile = None
if use_a_profile:
    import json
    print('\nUpload your PROFILE (.json):')
    up = colab_files.upload()
    profile = json.loads(pathlib.Path(list(up.keys())[0]).read_text(encoding='utf-8'))
    print(f'  ✅  Profile: {profile.get("name")} ({len(profile.get("entries") or {})} corrections)')


In [ ]:
#@title 🔑  Step 4 — Optional: let Claude read the report (NOT for confidential reports)
#@markdown Leave this off for confidential reports: the built-in reader is used and nothing is sent anywhere.
#@markdown Switching it on sends the report text to Anthropic's Claude API. Every value Claude returns must quote the report; unsupported values are discarded.
report_is_not_confidential_and_may_be_sent_to_claude = False #@param {type:"boolean"}
import os
from getpass import getpass
api_key = ''
if report_is_not_confidential_and_may_be_sent_to_claude:
    try:
        from google.colab import userdata
        api_key = userdata.get('ANTHROPIC_API_KEY') or ''
    except Exception:
        api_key = ''
    if not api_key:
        api_key = getpass('Paste your Claude API key: ').strip()
print('🤖  Claude will read the report.' if api_key else '🔒  Built-in reader only – the report stays in this session.')

In [ ]:
#@title 🔍  Step 5 — Read the report: what's accepted automatically and what needs a look
#@markdown Tick to also list the values that were accepted automatically:
show_accepted = False #@param {type:"boolean"}
import pandas as pd
print(f'📄  Reading {report_path.name} …')
text = faire.extract_report_text(report_path)
meta = faire.prepare(text, profile)
if api_key:
    try:
        faire.second_opinion(meta, faire.facts_from_claude(faire.parse_with_claude(text, api_key), text), 'Claude')
    except Exception as exc:
        print(f'⚠️   Claude could not be used ({exc}); using the built-in reader only.')
items = faire.review(meta, template_path)
c = faire.review_summary(items)
print(f"✅  {c['ok']} values accepted automatically · {c['check']} need a look · {c['missing']} not found "
      f"({c['missing_required']} of them mandatory/highly recommended) · {meta.get('_method', 'rule-based')}\n")
for n in meta.get('notes') or []:
    print('  •', n)
pd.set_option('display.max_colwidth', 120)
def table(rows):
    return pd.DataFrame([{'id': i['id'], 'field': i['term'], 'assay': i.get('assay') or ('all samples' if i['scope'] == 'samples' else ''),
                          'value': i['value'], 'why': '; '.join(i.get('reasons') or []), 'from the report': i.get('evidence') or ''} for i in rows])
print('\n🔎  NEEDS A LOOK'); display(table([i for i in items if i['status'] == 'check']))
print('\n❓  NOT FOUND (mandatory / highly recommended)'); display(table([i for i in items if i['status'] == 'missing' and i.get('required') in ('M', 'HR')]))
if show_accepted:
    print('\n✅  ACCEPTED AUTOMATICALLY'); display(table([i for i in items if i['status'] == 'ok' and not i['generated']]))


In [ ]:
#@title ✏️  Step 6 — Correct values (optional), fill the workbook and download it
#@markdown To change a value, add a line to `edits` using the **id** from the table above, e.g.
#@markdown `'sample:filter_material': 'mixed cellulose ester'`. Use `''` to mark a value as not stated.
edits = {
    # 'sample:filter_material': 'mixed cellulose ester',
}
#@markdown Tick to download your corrections as a profile, so the next report of this kind is corrected automatically:
save_profile = False #@param {type:"boolean"}
profile_name = "my FAIRe profile" #@param {type:"string"}
from google.colab import files as colab_files
import pathlib
faire.apply_review(meta, edits)
tmp_out = pathlib.Path('FAIRe_filled.xlsx')
summary = faire.fill_workbook(template_path, meta, report_path.name, tmp_out)
out_name = f"FAIRe_filled_{summary['projectId'] or 'project'}.xlsx"
tmp_out.rename(out_name)
print(f'''✅  Workbook filled: {out_name}
   Sites {summary["lakes"]} · assays {", ".join(summary["assays"])} · {summary["sampleRows"]} sample rows · {summary["experimentRows"]} run rows''')
colab_files.download(out_name)
for x in summary['validation']['problems']:
    print(f"⚠️   {x['sheet']} · {x['term']}: {x['what']} ({x['where']})")
for x in summary['validation']['fixed']:
    print(f"🔧  fixed {x['sheet']} · {x['term']}: {x['what']}")
if not summary['validation']['problems']:
    print('✅  Passes the FAIRe format, vocabulary and ID checks')
if save_profile and edits:
    import json, re
    prof = faire.profile_from_review(meta, edits, profile_name, profile)
    prof_file = re.sub(r'[^A-Za-z0-9_-]+', '_', profile_name) + '.faire-profile.json'
    pathlib.Path(prof_file).write_text(json.dumps(prof, indent=1, ensure_ascii=False), encoding='utf-8')
    colab_files.download(prof_file)
    print(f'💾  Profile saved: {prof_file}')
print('\nNext: open it in Excel, add anything still marked "missing: not provided", and validate at https://shiny.csiro.au/FAIRe-fier/')